In [1]:
# USGS metadata cleaning — dates & keys (UPDATED FOR DUAL FILE PROCESSING)
# -------------------------------------------------------------
# This script is meant to run in a Jupyter notebook. You can
# paste cells one-by-one, or run as a .py script. It:
#   - Loads TWO CSV files (original + additional)
#   - Creates a unique key id_page_number for each
#   - Builds cleaned coordinates (best-available lat/long)
#   - Standardizes water_type into fixed categories (with review column)
#   - Parses dates_of_recording into year_start / year_end
#   - Enforces the rule: ignore any year > 1980 and any
#     "present/current year/to date" open-ends
#   - Exports separate cleaned files + combined file + review files
# -------------------------------------------------------------

# %% Imports
import os
import re
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

# %% User paths (Windows)
DATA_DIR = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\original_data"
INPUT_CSV_MAIN = os.path.join(DATA_DIR, "cleaned_metadata_final.csv")
INPUT_CSV_ADDITIONAL = os.path.join(DATA_DIR, "final_sbmetadata.csv")

# Safe outputs alongside input (non-destructive)
OUTPUT_CSV_MAIN = os.path.join(DATA_DIR, "cleaned_metadata_with_years.csv")
OUTPUT_CSV_ADDITIONAL = os.path.join(DATA_DIR, "cleaned_sbmetadata_with_years.csv")
OUTPUT_CSV_COMBINED = os.path.join(DATA_DIR, "combined_metadata_with_years.csv")

REVIEW_CSV_MAIN = os.path.join(DATA_DIR, "dates_needs_review_main.csv")
REVIEW_CSV_ADDITIONAL = os.path.join(DATA_DIR, "dates_needs_review_additional.csv")
REVIEW_CSV_COMBINED = os.path.join(DATA_DIR, "dates_needs_review_combined.csv")

WATER_TYPE_REVIEW_CSV_MAIN = os.path.join(DATA_DIR, "water_type_needs_review_main.csv")
WATER_TYPE_REVIEW_CSV_ADDITIONAL = os.path.join(DATA_DIR, "water_type_needs_review_additional.csv")
WATER_TYPE_REVIEW_CSV_COMBINED = os.path.join(DATA_DIR, "water_type_needs_review_combined.csv")

# %% Helper functions (same as before)
def preview(dataframe, n=50, random=False, start=None, end=None):
    if random:
        return dataframe.sample(n).style
    elif start is not None and end is not None:
        return dataframe.loc[start:end].style
    else:
        return dataframe.head(n).style

def make_key(row: pd.Series) -> str:
    return f"{row['id']}_{row['page_number']}"

def _to_float_or_none(v: Any) -> Optional[float]:
    try:
        if v == '' or v is None:
            return None
        return float(str(v))
    except Exception:
        return None

def _normalize_text(s: str) -> str:
    s = s.replace('\u2013','-').replace('\u2014','-')
    s = re.sub(r"[\u2212\u2012\u2015]", "-", s)
    s = s.replace(';', ',')
    s = s.replace('\u00a0', ' ')
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def _expand_two_digit_end(start_year: int, end_two: int) -> int:
    base_century = start_year - (start_year % 100)
    candidate = base_century + end_two
    if end_two < (start_year % 100):
        candidate += 100
    return candidate

# Water type categorization
categories = {
    'groundwater': 'Groundwater',
    'stream discharge': 'Stream discharge',
    'precipitation': 'Precipitation',
    'spring': 'Springs',
    'reservoir': 'Reservoir',
    'water quality': 'Water quality',
    'irrigation': 'Irrigation',
    'not water related': 'Not water related',
    'other': 'Other'
}

def classify_water_type(raw: str) -> Tuple[Optional[str], Optional[str]]:
    if not raw or pd.isna(raw):
        return None, 'MISSING'
    s = raw.lower().strip()
    for key, label in categories.items():
        if key in s:
            return label, None
    return None, raw

# Date parsing constants and function
YEAR_MIN = 1800
YEAR_MAX = 1980

RE_YEAR4   = re.compile(r"\b(18\d{2}|19\d{2}|2000|20\d{2})\b")
RE_MMYYYY  = re.compile(r"\b(0?[1-9]|1[0-2])[-/](18\d{2}|19\d{2}|20\d{2})\b")
RE_YYYYMMDD= re.compile(r"\b(18\d{2}|19\d{2}|20\d{2})[-/](0?[1-9]|1[0-2])[-/]([0-2]?\d|3[01])\b")
RE_TWO_DIGIT_RANGE = re.compile(r"\b(18\d{2}|19\d{2})\s*[-/]\s*(\d{2})\b")

OPEN_TOKENS = (
    'present','to present','current','current year','to current year',
    'to date','t-','t/','tc','open','ongoing'
)

def parse_years(raw: Any) -> Dict[str, Any]:
    out = {'year_start': None,'year_end': None,'years_list': None,
           'date_parse_status': None,'date_parse_notes': None,'needs_date_review': False}

    if raw is None:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    s = str(raw).strip()
    if s == '' or s.lower() in {'not specified','-','--','n/a','na'}:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    norm = _normalize_text(s)
    norm_lower = norm.lower()
    open_ended = any(tok in norm_lower for tok in OPEN_TOKENS)

    years: List[int] = []

    for y in RE_YEAR4.findall(norm):
        try: years.append(int(y))
        except: pass
    for mm, yyyy in RE_MMYYYY.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for yyyy, mm, dd in RE_YYYYMMDD.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for start, end2 in RE_TWO_DIGIT_RANGE.findall(norm):
        try:
            s4 = int(start); e2 = int(end2)
            years.append(s4); years.append(_expand_two_digit_end(s4,e2))
        except: pass

    years = [y for y in years if YEAR_MIN <= y <= YEAR_MAX]
    if not years:
        out['date_parse_status'] = 'UNPARSED'; out['date_parse_notes'] = 'no_year_<=1980_found'; out['needs_date_review'] = True; return out

    years = sorted(set(years))
    out['years_list'] = years
    out['year_start'] = min(years)
    if open_ended:
        out['year_end'] = None; out['date_parse_status'] = 'OPEN_ENDED_IGNORED'; out['needs_date_review'] = True
    else:
        out['year_end'] = max(years)
        if len(years) == 1: out['date_parse_status'] = 'SINGLE_YEAR'
        elif len(years) == 2 and out['year_end'] != out['year_start']: out['date_parse_status'] = 'EXACT_RANGE'
        else: out['date_parse_status'] = 'MULTI_YEARS_OR_RANGES'
    return out

# %% Data processing function
def process_dataframe(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Process a single dataframe with all cleaning steps"""
    print(f"\n=== Processing {source_name} ===")
    print(f"Loaded {len(df):,} rows")
    
    # Basic hygiene
    required_cols = [
        'id','page_number','inferred_latitude','inferred_longitude',
        'actual_latitude','actual_longitude','location','townships_ranges_sections',
        'watersource_name','actual_county','inferred_county','dates_of_recording',
        'temporal_resolution','units_of_measurement','water_type','keyterms'
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected column(s) in {source_name}: {missing}")

    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Add source identifier
    df['data_source'] = source_name
    
    # Unique key
    df['id_page_number'] = df.apply(make_key, axis=1)

    # Coordinates
    actual_lat = df['actual_latitude'].map(_to_float_or_none)
    actual_lon = df['actual_longitude'].map(_to_float_or_none)
    inf_lat    = df['inferred_latitude'].map(_to_float_or_none)
    inf_lon    = df['inferred_longitude'].map(_to_float_or_none)

    best_lat = actual_lat.where(actual_lat.notna(), inf_lat)
    best_lon = actual_lon.where(actual_lon.notna(), inf_lon)

    df['latitude']  = best_lat
    df['longitude'] = best_lon

    # Water type categorization
    results = df['water_type'].apply(classify_water_type)
    df['water_type_clean'] = results.apply(lambda x: x[0])
    df['water_type_review'] = results.apply(lambda x: x[1])

    # Count entries with multiple water types
    multiple_types_count = 0
    for raw in df['water_type'].dropna():
        s = raw.lower().strip()
        matches = sum(1 for key in categories.keys() if key in s)
        if matches > 1:
            multiple_types_count += 1

    print(f"Entries with multiple water types: {multiple_types_count:,} out of {len(df[df['water_type'].notna()]):,} total entries with water_type data")

    # Date parsing
    df.rename(columns={'dates_of_recording': 'dates_of_recording_raw'}, inplace=True)
    parsed = df['dates_of_recording_raw'].apply(parse_years).apply(pd.Series)
    for col in parsed.columns:
        df[col] = parsed[col]

    df['year_start'] = df['year_start'].astype('Int64')
    df['year_end']   = df['year_end'].astype('Int64')

    n_unparsed = (df['date_parse_status'] == 'UNPARSED').sum()
    print(f"Rows with no parsable year: {n_unparsed:,}")

    # Date parsing failure analysis
    total_rows = len(df)
    empty_or_na = df['dates_of_recording_raw'].isna().sum()
    empty_strings = (df['dates_of_recording_raw'].astype(str).str.strip() == '').sum()
    standard_na_values = df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']).sum()

    total_missing = len(df[(df['dates_of_recording_raw'].isna()) | 
                          (df['dates_of_recording_raw'].astype(str).str.strip() == '') |
                          (df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']))])

    unparsed_with_content = len(df[(df['date_parse_status'] == 'UNPARSED') & 
                                  (~df['dates_of_recording_raw'].isna()) &
                                  (df['dates_of_recording_raw'].astype(str).str.strip() != '') &
                                  (~df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']))])

    print(f"\nDate parsing breakdown:")
    print(f"Total rows: {total_rows:,}")
    print(f"Empty/NA dates: {empty_or_na:,}")
    print(f"Empty strings: {empty_strings:,}")
    print(f"Standard NA values: {standard_na_values:,}")
    print(f"Total missing data: {total_missing:,}")
    print(f"Unparsed but has content: {unparsed_with_content:,}")
    print(f"Successfully parsed: {total_rows - n_unparsed:,}")
    
    return df

# %% Load and process both files
print("=== DUAL FILE PROCESSING ===")

# Load main file
df_main = pd.read_csv(INPUT_CSV_MAIN, dtype=str, keep_default_na=False)
df_main_processed = process_dataframe(df_main, "main")

# Load additional file
df_additional = pd.read_csv(INPUT_CSV_ADDITIONAL, dtype=str, keep_default_na=False)
df_additional_processed = process_dataframe(df_additional, "additional")

# Create combined dataframe
print(f"\n=== Creating Combined Dataset ===")
df_combined = pd.concat([df_main_processed, df_additional_processed], ignore_index=True)
print(f"Combined dataset: {len(df_combined):,} rows")
print(f"Main file contribution: {len(df_main_processed):,} rows")
print(f"Additional file contribution: {len(df_additional_processed):,} rows")

# Check for duplicate id_page_number keys across files
duplicate_keys = set(df_main_processed['id_page_number']) & set(df_additional_processed['id_page_number'])
if duplicate_keys:
    print(f"WARNING: {len(duplicate_keys)} duplicate id_page_number keys found across files!")
    print(f"First few duplicates: {list(duplicate_keys)[:5]}")
else:
    print("No duplicate id_page_number keys across files - good!")

# %% Export all files
def export_files(df_main, df_additional, df_combined):
    """Export all processed files and review files"""
    
    # Export main processed file
    try:
        df_main.to_csv(OUTPUT_CSV_MAIN, index=False)
        print(f"Exported main file: {OUTPUT_CSV_MAIN}")
    except Exception as e:
        print(f"Could not save {OUTPUT_CSV_MAIN}: {e}")
    
    # Export additional processed file
    try:
        df_additional.to_csv(OUTPUT_CSV_ADDITIONAL, index=False)
        print(f"Exported additional file: {OUTPUT_CSV_ADDITIONAL}")
    except Exception as e:
        print(f"Could not save {OUTPUT_CSV_ADDITIONAL}: {e}")
    
    # Export combined file
    try:
        df_combined.to_csv(OUTPUT_CSV_COMBINED, index=False)
        print(f"Exported combined file: {OUTPUT_CSV_COMBINED}")
    except Exception as e:
        print(f"Could not save {OUTPUT_CSV_COMBINED}: {e}")
    
    # Export review files for dates
    for df, review_path, name in [
        (df_main, REVIEW_CSV_MAIN, "main"),
        (df_additional, REVIEW_CSV_ADDITIONAL, "additional"), 
        (df_combined, REVIEW_CSV_COMBINED, "combined")
    ]:
        needs_review = df[df['needs_date_review'] == True]
        if not needs_review.empty:
            try:
                needs_review.to_csv(review_path, index=False)
                print(f"Exported {len(needs_review):,} rows needing date review ({name}) → {review_path}")
            except Exception as e:
                print(f"Could not save {review_path}: {e}")
    
    # Export review files for water types
    for df, review_path, name in [
        (df_main, WATER_TYPE_REVIEW_CSV_MAIN, "main"),
        (df_additional, WATER_TYPE_REVIEW_CSV_ADDITIONAL, "additional"),
        (df_combined, WATER_TYPE_REVIEW_CSV_COMBINED, "combined")
    ]:
        needs_wt_review = df[df['water_type_review'].notna()]
        if not needs_wt_review.empty:
            try:
                needs_wt_review.to_csv(review_path, index=False)
                print(f"Exported {len(needs_wt_review):,} rows needing water_type review ({name}) → {review_path}")
            except Exception as e:
                print(f"Could not save {review_path}: {e}")

export_files(df_main_processed, df_additional_processed, df_combined)

# %% Store processed dataframes for further analysis
df = df_combined  # For compatibility with existing analysis code
print(f"\n=== Files Ready for Analysis ===")
print("Available DataFrames:")
print(f"- df_main_processed: {df_main_processed.shape}")
print(f"- df_additional_processed: {df_additional_processed.shape}") 
print(f"- df_combined (also stored as 'df'): {df_combined.shape}")

=== DUAL FILE PROCESSING ===

=== Processing main ===
Loaded 107,314 rows
Entries with multiple water types: 251 out of 107,314 total entries with water_type data
Rows with no parsable year: 16,874

Date parsing breakdown:
Total rows: 107,314
Empty/NA dates: 0
Empty strings: 12,458
Standard NA values: 752
Total missing data: 13,210
Unparsed but has content: 3,664
Successfully parsed: 90,440

=== Processing additional ===
Loaded 2,312 rows
Entries with multiple water types: 9 out of 2,312 total entries with water_type data
Rows with no parsable year: 1,141

Date parsing breakdown:
Total rows: 2,312
Empty/NA dates: 0
Empty strings: 79
Standard NA values: 0
Total missing data: 79
Unparsed but has content: 1,062
Successfully parsed: 1,171

=== Creating Combined Dataset ===
Combined dataset: 109,626 rows
Main file contribution: 107,314 rows
Additional file contribution: 2,312 rows
No duplicate id_page_number keys across files - good!
Exported main file: C:\Users\aeliz\Dropbox\Documents\Jupy

In [ ]:
# %% Quick sanity checks for all datasets
def show_sample_analysis(df, name):
    print(f"\n=== {name} Sample Analysis ===")
    print("\nSample of parsed years:")
    print(df[['dates_of_recording_raw','year_start','year_end','date_parse_status']].head(12))
    print(f"\nUnique water_type_clean values:")
    print(df['water_type_clean'].dropna().unique())
    print(f"\nRows needing water_type_review:")
    review_sample = df[df['water_type_review'].notna()][['water_type','water_type_review']].head(10)
    if not review_sample.empty:
        print(review_sample)
    else:
        print("No water types need review")

# Show samples for all datasets
show_sample_analysis(df_main_processed, "MAIN FILE")
show_sample_analysis(df_additional_processed, "ADDITIONAL FILE") 
show_sample_analysis(df_combined, "COMBINED DATASET")

# %% Generate summaries for all datasets
def generate_summaries(df, name):
    """Generate summaries by water type, year, and decade for a dataset"""
    print(f"\n=== {name} SUMMARIES ===")
    
    # Exclude rows where water_type_clean is missing
    df_summary = df[df['water_type_clean'].notna()].copy()
    
    print(f"Dataset size: {len(df):,} total rows, {len(df_summary):,} with water_type_clean")
    
    # By water type only
    summary_by_type = (
        df_summary
        .groupby('water_type_clean')
        .agg(
            n_rows=('id_page_number','size'),
            n_pages=('id_page_number','nunique'),
            n_docs=('id','nunique')
        )
        .reset_index()
        .sort_values('n_rows', ascending=False)
    )
    print(f"\nSummary by water type:")
    print(summary_by_type)
    
    # By water type × year (with year_start data)
    df_with_years = df_summary[df_summary['year_start'].notna()].copy()
    summary_by_year = (
        df_with_years
        .groupby(['water_type_clean','year_start'])
        .agg(
            n_rows=('id_page_number','size'),
            n_pages=('id_page_number','nunique'),
            n_docs=('id','nunique')
        )
        .reset_index()
        .sort_values(['water_type_clean','year_start'])
    )
    print(f"\nSummary by water type × year (showing first 20 rows):")
    print(summary_by_year.head(20))
    
    # By water type × decade
    df_with_years['decade'] = (df_with_years['year_start'] // 10) * 10
    summary_by_decade = (
        df_with_years
        .groupby(['water_type_clean','decade'])
        .agg(
            n_rows=('id_page_number','size'),
            n_pages=('id_page_number','nunique'),
            n_docs=('id','nunique')
        )
        .reset_index()
        .sort_values(['water_type_clean','decade'])
    )
    print(f"\nSummary by water type × decade:")
    print(summary_by_decade)
    
    return {
        'summary_by_type': summary_by_type,
        'summary_by_year': summary_by_year, 
        'summary_by_decade': summary_by_decade
    }

# Generate summaries for all datasets
main_summaries = generate_summaries(df_main_processed, "MAIN FILE")
additional_summaries = generate_summaries(df_additional_processed, "ADDITIONAL FILE")
combined_summaries = generate_summaries(df_combined, "COMBINED DATASET")

# Store summary tables with descriptive names
summary_by_type = combined_summaries['summary_by_type']
summary_by_year = combined_summaries['summary_by_year']
summary_by_decade = combined_summaries['summary_by_decade']

# Also store individual file summaries
main_summary_by_type = main_summaries['summary_by_type']
main_summary_by_year = main_summaries['summary_by_year']
main_summary_by_decade = main_summaries['summary_by_decade']

additional_summary_by_type = additional_summaries['summary_by_type']
additional_summary_by_year = additional_summaries['summary_by_year']
additional_summary_by_decade = additional_summaries['summary_by_decade']

# %% Comparative analysis
print(f"\n=== COMPARATIVE ANALYSIS ===")

# Compare data source contributions
print("Data source breakdown in combined dataset:")
source_breakdown = df_combined['data_source'].value_counts()
print(source_breakdown)
print(f"Percentage breakdown:")
for source, count in source_breakdown.items():
    print(f"  {source}: {count:,} rows ({100*count/len(df_combined):.1f}%)")

# Compare water type distributions
print(f"\nWater type comparison:")
print("Main file top water types:")
print(main_summary_by_type.head())
print(f"\nAdditional file top water types:")
print(additional_summary_by_type.head())

# Year range comparison
main_years = df_main_processed[df_main_processed['year_start'].notna()]['year_start']
additional_years = df_additional_processed[df_additional_processed['year_start'].notna()]['year_start']

print(f"\nYear range comparison:")
print(f"Main file: {main_years.min()}-{main_years.max()} (n={len(main_years):,})")
print(f"Additional file: {additional_years.min()}-{additional_years.max()} (n={len(additional_years):,})")

print(f"\n=== PROCESSING COMPLETE ===")
print("Available DataFrames for further analysis:")
for name in ['df_main_processed', 'df_additional_processed', 'df_combined', 'df']:
    if name in globals():
        df_obj = globals()[name]
        print(f"- {name}: {df_obj.shape}")

print("\nAvailable summary tables:")
summary_tables = [name for name in globals() if 'summary' in name.lower() and isinstance(globals()[name], pd.DataFrame)]
for name in summary_tables:
    print(f"- {name}: {globals()[name].shape}")

## Addition 9/1

In [2]:
# USGS metadata cleaning — dates & keys (UPDATED FOR DUAL FILE PROCESSING)
# -------------------------------------------------------------
# This script is meant to run in a Jupyter notebook. You can
# paste cells one-by-one, or run as a .py script. It:
#   - Loads TWO CSV files (original + additional)
#   - Creates a unique key id_page_number for each
#   - Builds cleaned coordinates (best-available lat/long)
#   - Standardizes water_type into fixed categories (with review column)
#   - Parses dates_of_recording into year_start / year_end
#   - Enforces the rule: ignore any year > 1980 and any
#     "present/current year/to date" open-ends
#   - Exports separate cleaned files + combined file + review files
# -------------------------------------------------------------

# %% Imports
import os
import re
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

# %% User paths (Windows)
DATA_DIR = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\original_data"
INPUT_CSV_MAIN = os.path.join(DATA_DIR, "cleaned_metadata_final.csv")
INPUT_CSV_ADDITIONAL = os.path.join(DATA_DIR, "final_sbmetadata.csv")

# Safe outputs alongside input (non-destructive)
OUTPUT_CSV_MAIN = os.path.join(DATA_DIR, "cleaned_metadata_with_years.csv")
OUTPUT_CSV_ADDITIONAL = os.path.join(DATA_DIR, "cleaned_sbmetadata_with_years.csv")
OUTPUT_CSV_COMBINED = os.path.join(DATA_DIR, "combined_metadata_with_years.csv")

REVIEW_CSV_MAIN = os.path.join(DATA_DIR, "dates_needs_review_main.csv")
REVIEW_CSV_ADDITIONAL = os.path.join(DATA_DIR, "dates_needs_review_additional.csv")
REVIEW_CSV_COMBINED = os.path.join(DATA_DIR, "dates_needs_review_combined.csv")

WATER_TYPE_REVIEW_CSV_MAIN = os.path.join(DATA_DIR, "water_type_needs_review_main.csv")
WATER_TYPE_REVIEW_CSV_ADDITIONAL = os.path.join(DATA_DIR, "water_type_needs_review_additional.csv")
WATER_TYPE_REVIEW_CSV_COMBINED = os.path.join(DATA_DIR, "water_type_needs_review_combined.csv")

# %% Helper functions (same as before)
def preview(dataframe, n=50, random=False, start=None, end=None):
    if random:
        return dataframe.sample(n).style
    elif start is not None and end is not None:
        return dataframe.loc[start:end].style
    else:
        return dataframe.head(n).style

def make_key(row: pd.Series) -> str:
    return f"{row['id']}_{row['page_number']}"

def _to_float_or_none(v: Any) -> Optional[float]:
    try:
        if v == '' or v is None:
            return None
        return float(str(v))
    except Exception:
        return None

def _normalize_text(s: str) -> str:
    s = s.replace('\u2013','-').replace('\u2014','-')
    s = re.sub(r"[\u2212\u2012\u2015]", "-", s)
    s = s.replace(';', ',')
    s = s.replace('\u00a0', ' ')
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def _expand_two_digit_end(start_year: int, end_two: int) -> int:
    base_century = start_year - (start_year % 100)
    candidate = base_century + end_two
    if end_two < (start_year % 100):
        candidate += 100
    return candidate

# Water type categorization
categories = {
    'groundwater': 'Groundwater',
    'stream discharge': 'Stream discharge',
    'precipitation': 'Precipitation',
    'spring': 'Springs',
    'reservoir': 'Reservoir',
    'water quality': 'Water quality',
    'irrigation': 'Irrigation',
    'not water related': 'Not water related',
    'other': 'Other'
}

def classify_water_type(raw: str) -> Tuple[Optional[str], Optional[str]]:
    if not raw or pd.isna(raw):
        return None, 'MISSING'
    s = raw.lower().strip()
    for key, label in categories.items():
        if key in s:
            return label, None
    return None, raw

# Date parsing constants and function
YEAR_MIN = 1800
YEAR_MAX = 1980

RE_YEAR4   = re.compile(r"\b(18\d{2}|19\d{2}|2000|20\d{2})\b")
RE_MMYYYY  = re.compile(r"\b(0?[1-9]|1[0-2])[-/](18\d{2}|19\d{2}|20\d{2})\b")
RE_YYYYMMDD= re.compile(r"\b(18\d{2}|19\d{2}|20\d{2})[-/](0?[1-9]|1[0-2])[-/]([0-2]?\d|3[01])\b")
RE_TWO_DIGIT_RANGE = re.compile(r"\b(18\d{2}|19\d{2})\s*[-/]\s*(\d{2})\b")

OPEN_TOKENS = (
    'present','to present','current','current year','to current year',
    'to date','t-','t/','tc','open','ongoing'
)

def parse_years(raw: Any) -> Dict[str, Any]:
    out = {'year_start': None,'year_end': None,'years_list': None,
           'date_parse_status': None,'date_parse_notes': None,'needs_date_review': False}

    if raw is None:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    s = str(raw).strip()
    if s == '' or s.lower() in {'not specified','-','--','n/a','na'}:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    norm = _normalize_text(s)
    norm_lower = norm.lower()
    open_ended = any(tok in norm_lower for tok in OPEN_TOKENS)

    years: List[int] = []

    for y in RE_YEAR4.findall(norm):
        try: years.append(int(y))
        except: pass
    for mm, yyyy in RE_MMYYYY.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for yyyy, mm, dd in RE_YYYYMMDD.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for start, end2 in RE_TWO_DIGIT_RANGE.findall(norm):
        try:
            s4 = int(start); e2 = int(end2)
            years.append(s4); years.append(_expand_two_digit_end(s4,e2))
        except: pass

    years = [y for y in years if YEAR_MIN <= y <= YEAR_MAX]
    if not years:
        out['date_parse_status'] = 'UNPARSED'; out['date_parse_notes'] = 'no_year_<=1980_found'; out['needs_date_review'] = True; return out

    years = sorted(set(years))
    out['years_list'] = years
    out['year_start'] = min(years)
    if open_ended:
        out['year_end'] = None; out['date_parse_status'] = 'OPEN_ENDED_IGNORED'; out['needs_date_review'] = True
    else:
        out['year_end'] = max(years)
        if len(years) == 1: out['date_parse_status'] = 'SINGLE_YEAR'
        elif len(years) == 2 and out['year_end'] != out['year_start']: out['date_parse_status'] = 'EXACT_RANGE'
        else: out['date_parse_status'] = 'MULTI_YEARS_OR_RANGES'
    return out

# %% Data processing function
def process_dataframe(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Process a single dataframe with all cleaning steps"""
    print(f"\n=== Processing {source_name} ===")
    print(f"Loaded {len(df):,} rows")
    
    # Basic hygiene
    required_cols = [
        'id','page_number','inferred_latitude','inferred_longitude',
        'actual_latitude','actual_longitude','location','townships_ranges_sections',
        'watersource_name','actual_county','inferred_county','dates_of_recording',
        'temporal_resolution','units_of_measurement','water_type','keyterms'
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected column(s) in {source_name}: {missing}")

    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Add source identifier
    df['data_source'] = source_name
    
    # Unique key
    df['id_page_number'] = df.apply(make_key, axis=1)

    # Coordinates
    actual_lat = df['actual_latitude'].map(_to_float_or_none)
    actual_lon = df['actual_longitude'].map(_to_float_or_none)
    inf_lat    = df['inferred_latitude'].map(_to_float_or_none)
    inf_lon    = df['inferred_longitude'].map(_to_float_or_none)

    best_lat = actual_lat.where(actual_lat.notna(), inf_lat)
    best_lon = actual_lon.where(actual_lon.notna(), inf_lon)

    df['latitude']  = best_lat
    df['longitude'] = best_lon

    # Water type categorization
    results = df['water_type'].apply(classify_water_type)
    df['water_type_clean'] = results.apply(lambda x: x[0])
    df['water_type_review'] = results.apply(lambda x: x[1])

    # Count entries with multiple water types
    multiple_types_count = 0
    for raw in df['water_type'].dropna():
        s = raw.lower().strip()
        matches = sum(1 for key in categories.keys() if key in s)
        if matches > 1:
            multiple_types_count += 1

    print(f"Entries with multiple water types: {multiple_types_count:,} out of {len(df[df['water_type'].notna()]):,} total entries with water_type data")

    # Date parsing
    df.rename(columns={'dates_of_recording': 'dates_of_recording_raw'}, inplace=True)
    parsed = df['dates_of_recording_raw'].apply(parse_years).apply(pd.Series)
    for col in parsed.columns:
        df[col] = parsed[col]

    df['year_start'] = df['year_start'].astype('Int64')
    df['year_end']   = df['year_end'].astype('Int64')

    n_unparsed = (df['date_parse_status'] == 'UNPARSED').sum()
    print(f"Rows with no parsable year: {n_unparsed:,}")

    # Date parsing failure analysis
    total_rows = len(df)
    empty_or_na = df['dates_of_recording_raw'].isna().sum()
    empty_strings = (df['dates_of_recording_raw'].astype(str).str.strip() == '').sum()
    standard_na_values = df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']).sum()

    total_missing = len(df[(df['dates_of_recording_raw'].isna()) | 
                          (df['dates_of_recording_raw'].astype(str).str.strip() == '') |
                          (df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']))])

    unparsed_with_content = len(df[(df['date_parse_status'] == 'UNPARSED') & 
                                  (~df['dates_of_recording_raw'].isna()) &
                                  (df['dates_of_recording_raw'].astype(str).str.strip() != '') &
                                  (~df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']))])

    print(f"\nDate parsing breakdown:")
    print(f"Total rows: {total_rows:,}")
    print(f"Empty/NA dates: {empty_or_na:,}")
    print(f"Empty strings: {empty_strings:,}")
    print(f"Standard NA values: {standard_na_values:,}")
    print(f"Total missing data: {total_missing:,}")
    print(f"Unparsed but has content: {unparsed_with_content:,}")
    print(f"Successfully parsed: {total_rows - n_unparsed:,}")
    
    return df

# %% Load and process both files
print("=== DUAL FILE PROCESSING ===")

# Load main file
df_main = pd.read_csv(INPUT_CSV_MAIN, dtype=str, keep_default_na=False)
df_main_processed = process_dataframe(df_main, "main")

# Load additional file
df_additional = pd.read_csv(INPUT_CSV_ADDITIONAL, dtype=str, keep_default_na=False)
df_additional_processed = process_dataframe(df_additional, "additional")

# Create combined dataframe
print(f"\n=== Creating Combined Dataset ===")
df_combined = pd.concat([df_main_processed, df_additional_processed], ignore_index=True)
print(f"Combined dataset: {len(df_combined):,} rows")
print(f"Main file contribution: {len(df_main_processed):,} rows")
print(f"Additional file contribution: {len(df_additional_processed):,} rows")

# Check for duplicate id_page_number keys across files
duplicate_keys = set(df_main_processed['id_page_number']) & set(df_additional_processed['id_page_number'])
if duplicate_keys:
    print(f"WARNING: {len(duplicate_keys)} duplicate id_page_number keys found across files!")
    print(f"First few duplicates: {list(duplicate_keys)[:5]}")
else:
    print("No duplicate id_page_number keys across files - good!")

# %% Export all files
def export_files(df_main, df_additional, df_combined):
    """Export all processed files and review files"""
    
    # Export main processed file
    try:
        df_main.to_csv(OUTPUT_CSV_MAIN, index=False)
        print(f"Exported main file: {OUTPUT_CSV_MAIN}")
    except Exception as e:
        print(f"Could not save {OUTPUT_CSV_MAIN}: {e}")
    
    # Export additional processed file
    try:
        df_additional.to_csv(OUTPUT_CSV_ADDITIONAL, index=False)
        print(f"Exported additional file: {OUTPUT_CSV_ADDITIONAL}")
    except Exception as e:
        print(f"Could not save {OUTPUT_CSV_ADDITIONAL}: {e}")
    
    # Export combined file
    try:
        df_combined.to_csv(OUTPUT_CSV_COMBINED, index=False)
        print(f"Exported combined file: {OUTPUT_CSV_COMBINED}")
    except Exception as e:
        print(f"Could not save {OUTPUT_CSV_COMBINED}: {e}")
    
    # Export review files for dates
    for df, review_path, name in [
        (df_main, REVIEW_CSV_MAIN, "main"),
        (df_additional, REVIEW_CSV_ADDITIONAL, "additional"), 
        (df_combined, REVIEW_CSV_COMBINED, "combined")
    ]:
        needs_review = df[df['needs_date_review'] == True]
        if not needs_review.empty:
            try:
                needs_review.to_csv(review_path, index=False)
                print(f"Exported {len(needs_review):,} rows needing date review ({name}) → {review_path}")
            except Exception as e:
                print(f"Could not save {review_path}: {e}")
    
    # Export review files for water types
    for df, review_path, name in [
        (df_main, WATER_TYPE_REVIEW_CSV_MAIN, "main"),
        (df_additional, WATER_TYPE_REVIEW_CSV_ADDITIONAL, "additional"),
        (df_combined, WATER_TYPE_REVIEW_CSV_COMBINED, "combined")
    ]:
        needs_wt_review = df[df['water_type_review'].notna()]
        if not needs_wt_review.empty:
            try:
                needs_wt_review.to_csv(review_path, index=False)
                print(f"Exported {len(needs_wt_review):,} rows needing water_type review ({name}) → {review_path}")
            except Exception as e:
                print(f"Could not save {review_path}: {e}")

export_files(df_main_processed, df_additional_processed, df_combined)

# %% Discover DataFrames in memory and save summaries + full table
import pandas as pd
from artifacts_io import save_artifacts

# 1) Collect all pandas DataFrames alive in the current notebook
dfs_in_mem = {name: obj for name, obj in globals().items() if isinstance(obj, pd.DataFrame)}

if not dfs_in_mem:
    raise RuntimeError("No pandas DataFrames found in memory. Run the upstream analysis cells first.")

print("Found DataFrames:")
for k, v in sorted(dfs_in_mem.items()):
    print(f"  - {k:28s} shape={v.shape} cols={list(v.columns)[:6]}{'...' if v.shape[1]>6 else ''}")

# 2) Heuristics for "summary" tables: common naming patterns & presence of groupby-like columns
name_hits = ("summary", "sum_", "_sum", "agg", "group", "pivot", "tbl", "by_", "_by")
summary_like = [k for k in dfs_in_mem if any(tok in k.lower() for tok in name_hits)]

# If nothing matched by name, look for "tall-to-wide" hints (few rows, many categorical columns)
if not summary_like:
    for k, df in dfs_in_mem.items():
        if df.shape[0] < max(200, 0.05 * max(d.shape[0] for d in dfs_in_mem.values())):
            summary_like.append(k)

# 3) Heuristic for "everything" table: largest number of rows
everything_key = max(dfs_in_mem, key=lambda k: dfs_in_mem[k].shape[0])

print("\nAuto-identified candidates:")
print("  Summary-like tables:", summary_like if summary_like else "(none found by heuristics)")
print("  Everything table    :", everything_key, dfs_in_mem[everything_key].shape)

# 4) For dual-file processing, ensure key datasets are included
key_datasets = ['df_main_processed', 'df_additional_processed', 'df_combined']
summary_like.extend([k for k in key_datasets if k in dfs_in_mem and k not in summary_like])

# Build what to save
dfs_to_save = {}
for key in summary_like:
    if key in dfs_in_mem:
        dfs_to_save[key] = dfs_in_mem[key]

# Ensure the main datasets are included
dfs_to_save[everything_key] = dfs_in_mem[everything_key]   # ensure the full table is included

# Add any remaining key datasets that weren't captured
for key in key_datasets:
    if key in dfs_in_mem and key not in dfs_to_save:
        dfs_to_save[key] = dfs_in_mem[key]

meta = {
    "notes": "Dual-file processing: main + additional Santa Barbara datasets",
    "source_notebook": "working_datacrunch_sbaddition.ipynb", 
    "summary_keys": summary_like,
    "everything_key": everything_key,
    "processing_type": "dual_file_with_sb_additional",
    "datasets": {
        "main": "cleaned_metadata_final.csv",
        "additional": "final_sbmetadata.csv", 
        "combined": "combined_metadata_with_years.csv"
    }
}

run_dir = save_artifacts(dfs_to_save, meta=meta, out_dir="artifacts")
print("\nSaved artifacts to:", run_dir)

# %% Store processed dataframes for further analysis
df = df_combined  # For compatibility with existing analysis code
print(f"\n=== Files Ready for Analysis ===")
print("Available DataFrames:")
print(f"- df_main_processed: {df_main_processed.shape}")
print(f"- df_additional_processed: {df_additional_processed.shape}") 
print(f"- df_combined (also stored as 'df'): {df_combined.shape}")

print(f"\nArtifacts exported containing:")
for name, df_obj in dfs_to_save.items():
    print(f"- {name}: {df_obj.shape}")

=== DUAL FILE PROCESSING ===

=== Processing main ===
Loaded 107,314 rows
Entries with multiple water types: 251 out of 107,314 total entries with water_type data
Rows with no parsable year: 16,874

Date parsing breakdown:
Total rows: 107,314
Empty/NA dates: 0
Empty strings: 12,458
Standard NA values: 752
Total missing data: 13,210
Unparsed but has content: 3,664
Successfully parsed: 90,440

=== Processing additional ===
Loaded 2,312 rows
Entries with multiple water types: 9 out of 2,312 total entries with water_type data
Rows with no parsable year: 1,141

Date parsing breakdown:
Total rows: 2,312
Empty/NA dates: 0
Empty strings: 79
Standard NA values: 0
Total missing data: 79
Unparsed but has content: 1,062
Successfully parsed: 1,171

=== Creating Combined Dataset ===
Combined dataset: 109,626 rows
Main file contribution: 107,314 rows
Additional file contribution: 2,312 rows
No duplicate id_page_number keys across files - good!
Exported main file: C:\Users\aeliz\Dropbox\Documents\Jupy